## Client

In [9]:
import os
from dotenv import load_dotenv
from datapizzai.clients import ClientFactory
from datapizzai.type import TextBlock

# Carica variabili da .env (root progetto)
load_dotenv()

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5",
    temperature=1
)

# Invoke sempplice (minimale)
print(client.invoke("Ciao, piacere di conoscerti").text)

# Oppure usando un modulo che vedremo tra un attimo
print(client.invoke(TextBlock(content="Ciao, piacere di conoscerti")).text)

Ciao! Piacere mio. Come posso aiutarti oggi?
Ciao! Piacere di conoscerti anche per me. Sono il tuo assistente: come posso aiutarti oggi?


In [29]:
import requests
from typing import Optional, Union, List
from pydantic import BaseModel

from datapizzai.type import TextBlock
from datapizzai.memory import Memory


class SimpleResponse(BaseModel):
    text: str
    prompt_tokens_used: int = 0
    completion_tokens_used: int = 0
    stop_reason: str = "stop"


class OllamaClient:
    def __init__(self, model: str = "gemma3n:e2b", base_url: str = "http://localhost:11434"):
        self.model = model
        self.base_url = base_url.rstrip("/")

    def _build_messages(self, input=None, memory: Optional[Memory] = None):
        msgs = []
        if memory is not None:
            for turn in memory.memory:
                role = turn.role.value if hasattr(turn.role, "value") else str(turn.role)
                content = " ".join(getattr(b, "content", "") for b in turn.blocks)
                if content:
                    msgs.append({"role": role, "content": content})
        if isinstance(input, str) and input:
            msgs.append({"role": "user", "content": input})
        return msgs

    def invoke(self, input=None, memory: Optional[Memory] = None) -> SimpleResponse:
        payload = {"model": self.model, "messages": self._build_messages(input, memory), "stream": False}
        try:
            r = requests.post(f"{self.base_url}/api/chat", json=payload, timeout=120)
            r.raise_for_status()
            data = r.json()
            text = data.get("message", {}).get("content") or str(data)
        except Exception as e:
            text = f"Errore Ollama: {e}"
        return SimpleResponse(text=text)


if __name__ == "__main__":
    client = OllamaClient()
    print(client.invoke("Ciao! Riassumi in una frase il teorema di Pitagora.").text)

Il teorema di Pitagora afferma che in un triangolo rettangolo, il quadrato costruito sull'ipotenusa (il lato opposto all'angolo retto) è uguale alla somma dei quadrati costruiti sui due cateti (i lati che formano l'angolo retto).



In [31]:
from datapizzai.memory import Memory
from datapizzai.type import TextBlock, ROLE

class SummarizingChat:
    def __init__(self, client, summarize_every: int = 5, max_summary_len: int = 6):
        self.client = client
        self.memory = Memory()
        self.turns = 0
        self.summarize_every = summarize_every
        self.max_summary_len = max_summary_len

    def _summarize(self):
        # Chiede al modello un riassunto della conversazione corrente
        prompt = (
            f"Riassumi la conversazione in {self.max_summary_len} frasi, "
            "mettendo in evidenza decisioni e TODO."
        )
        summary_resp = self.client.invoke(prompt, memory=self.memory)
        summary = summary_resp.text.strip()
        # Resetta la memoria mantenendo solo il riassunto come punto di partenza
        new_mem = Memory()
        new_mem.add_turn([TextBlock(content=f"[Riassunto] {summary}")], ROLE.ASSISTANT)
        self.memory = new_mem

    def send(self, user_input: str) -> str:
        self.memory.add_turn([TextBlock(content=user_input)], ROLE.USER)
        resp = self.client.invoke("", memory=self.memory)
        self.memory.add_turn([TextBlock(content=resp.text)], ROLE.ASSISTANT)
        self.turns += 1
        if self.turns % self.summarize_every == 0:
            self._summarize()
        return resp.text

# Uso:
chat = SummarizingChat(client, summarize_every=5)
print(chat.send("Iniziamo a progettare una REST API per un e‑commerce."))

Ottimo! Iniziamo a progettare la REST API per un e-commerce. Per rendere il processo il più efficace possibile, strutturiamo la progettazione in fasi e considerazioni chiave.

**Fase 1: Identificazione dei Requisiti e degli Utenti**

Prima di scrivere una sola riga di codice, è fondamentale capire chi utilizzerà l'API e cosa deve fare.

*   **Utenti Principali:**
    *   **Client (App Mobile/Web):**  Gli utenti finali che navigano e acquistano prodotti.
    *   **Admin (Gestori):**  Gli utenti che gestiscono il catalogo prodotti, gli ordini, i clienti, ecc.
    *   **Sistema di Pagamento:**  L'integrazione con un gateway di pagamento (Stripe, PayPal, ecc.).
    *   **Sistema di Notifica:**  Un sistema per inviare email/SMS agli utenti (es. notifiche di conferma ordine, spedizione).
*   **Funzionalità Chiave:**
    *   **Gestione Prodotti:** Creazione, lettura, aggiornamento, eliminazione (CRUD).
    *   **Gestione Clienti:** Creazione, lettura, aggiornamento, eliminazione.
    *   **Ge

In [22]:
from datapizzai.cache import MemoryCache
import time

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5",
    temperature=1,
    cache=MemoryCache(),  # cache in-memory
)

# Stessa richiesta 2 volte: la seconda dovrebbe colpire la cache
q = "Dimmi 3 vantaggi del TDD in 1 riga"

t0 = time.perf_counter()
r1 = client.invoke(q)
t1 = time.perf_counter()
print("prima:", r1.text)
print(f"⏱️ tempo (prima): {t1 - t0:.3f}s")

t2 = time.perf_counter()
r2 = client.invoke(q)  # Qui avviene un cache hit, il client non viene invocato
t3 = time.perf_counter()
print("seconda:", r2.text)
print(f"⏱️ tempo (seconda): {t3 - t2:.3f}s")

prima: Feedback rapido; design più pulito e manutenibile; riduzione delle regressioni.
⏱️ tempo (prima): 6.340s
INFO     [2025-08-28 17:40:59 - datapizzai.cache.cache:109] Cache hit for 549d7d9fea78e1487f6a5993a57e7e217f9fd963d60bfb58febfbc0028fd5dec
seconda: Feedback rapido; design più pulito e manutenibile; riduzione delle regressioni.
⏱️ tempo (seconda): 0.000s


In [16]:
import os
from datapizzai.clients import ClientFactory
from datapizzai.memory import Memory
from datapizzai.type import TextBlock, ROLE

class Chatbot:
    def __init__(self, client, window_size: int = 6):
        self.client = client
        self.memory = Memory()
        self.window_size = window_size

    def _apply_sliding_window(self):
        if len(self.memory.memory) > self.window_size:
            self.memory.memory = self.memory.memory[-self.window_size:]

    def send(self, user_input: str) -> str:
        self.memory.add_turn([TextBlock(content=user_input)], ROLE.USER)
        self._apply_sliding_window()
        response = self.client.invoke("", memory=self.memory)
        self.memory.add_turn([TextBlock(content=response.text)], ROLE.ASSISTANT)
        # Stampa metriche minime (opzionale)
        total_tokens = (response.prompt_tokens_used or 0) + (response.completion_tokens_used or 0)
        print(f"[metriche] token totali: {total_tokens}")
        return response.text

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5",
    temperature=1,
)

bot = Chatbot(client, window_size=6)
print("Chat pronta. Digita 'esci' per terminare.")
while True:
    try:
        user = input("tu> ").strip()
        if user.lower() in {"esci", "exit", "quit"}:
            break
        print("bot>", bot.send(user))
    except KeyboardInterrupt:
        break
    except Exception:
        print("bot> Si è verificato un errore temporaneo. Riprova.")

Chat pronta. Digita 'esci' per terminare.


tu>  bella broooo


[metriche] token totali: 360
bot> Bellaaa! Dimmi pure, in cosa posso darti una mano? Preferisci italiano o inglese?


tu>  Ma quanto sei forte?


[metriche] token totali: 393
bot> Dipende da cosa ti serve! Sono “forte” nel:
- Spiegare concetti complessi in modo semplice
- Riassumere e tradurre testi
- Scrivere, correggere e migliorare contenuti
- Brainstorming di idee creative
- Aiutare con coding e debug

Niente panca piana però! Su cosa vuoi mettermi alla prova?


tu>  exit


In [19]:
from datapizzai.clients import ClientFactory
from datapizzai.type import TextBlock, MediaBlock, Media
from dotenv import load_dotenv
import os

load_dotenv('../.env')
client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-4o"
)

# Analizza immagine da URL
media = Media(
    extension="jpg",        # Estensione senza punto per MIME type corretto
    media_type="image",     # Tipo di media (image, audio, video)
    source_type="url",      # Fonte: url, base64, o file
    source="https://assets.science.nasa.gov/dynamicimage/assets/science/psd/mars/internal_resources/1155.jpeg?w=1767&h=350&fit=clip&crop=faces%2Cfocalpoint"
)

# Combina testo e immagine per input multimodale
response = client.invoke([
    TextBlock(content="Descrivi questa immagine in dettaglio"),
    MediaBlock(media=media)  # Wrapper che contiene l'immagine
])

print(response.text)

L'immagine è un panorama di un paesaggio marziano, probabilmente catturato da un rover della NASA. Presenta una vasta distesa di terreno roccioso e sabbioso con diverse formazioni geologiche. Sono visibili delle etichette che identificano particolari aree o punti di interesse, come "delta front", "Faillefeu", "Santa Cruz", "Marte" e "Rover Route".

Il terreno appare irregolare, con rocce sparse di varie dimensioni. L'orizzonte è leggermente ondulato e ci sono leggere variazioni di colore nel terreno, suggerendo differenti composizioni minerali o geologiche. L'immagine è suddivisa in sezioni numerate, probabilmente utilizzate per orientamento o riferimento scientifico. Lo sfondo è nero, evidenziando la superficie marziana.


In [21]:
import base64
from pathlib import Path
from datapizzai.type import Media, MediaBlock, TextBlock

def load_image_as_base64(path: str) -> str:
    """Converte file immagine in stringa base64 per trasmissione sicura"""
    return base64.b64encode(Path(path).read_bytes()).decode("utf-8")

# Carica immagine locale e converti in base64
image_b64 = load_image_as_base64("Client/Example.png")

# Crea oggetto Media con metadati dell'immagine
media = Media(
    extension="jpg",        # Estensione file per MIME type
    media_type="image",     # Tipo di contenuto
    source_type="base64",   # Formato di trasmissione
    source=image_b64,       # Dati immagine codificati
    detail="high"           # Qualità analisi (high per dettagli)
)

# Prompt specifico per analisi tecnica
prompt = "Analizza questa immagine e dammi una descrizione tecnica."

# Invoca AI con input multimodale (testo + immagine)
response = client.invoke([
    TextBlock(content=prompt),
    MediaBlock(media=media)  # Wrapper per l'immagine
])

print(response.content)

[TextBlock(content=L'immagine mostra un gruppo di persone sedute attorno a un tavolo, ciascuna con un laptop o un computer desktop. Sono impegnate in attività lavorative, probabilmente legate all'informatica o allo sviluppo software. Sul tavolo ci sono diverse lattine di bevande energetiche e una scatola di pizza aperta, con diverse fette già servite su piatti di carta. Ci sono anche fogli sparsi e post-it, suggerendo un ambiente di lavoro collaborativo e dinamico. L'illuminazione è fornita da una lampada da tavolo e l'atmosfera sembra serale o notturna. Su uno dei laptop è visibile il logo di Fedora, un sistema operativo basato su Linux.)]


In [27]:
from pathlib import Path
from datapizzai.type import Media, MediaBlock, TextBlock

analysis_client_google = client = ClientFactory.create(
            model="gemini-2.5-flash",
            provider="google",
            api_key=os.getenv("GOOGLE_API_KEY"),
            system_prompt="Sei un assistente AI esperto nell'analisi di audio. Rispondi in italiano.",
            temperature=0.5
        )

media = Media(
    extension="wav",
    media_type="audio",
    source_type="path",
    source="Client/TI0TpOD_.wav"        # <— percorso al file
)

prompt = "Trascrivi questo audio e riassumi il contenuto principale."
response = analysis_client_google.invoke([TextBlock(content=prompt), MediaBlock(media=media)])
print(response.content)

[TextBlock(content=Ecco la trascrizione e il riassunto dell'audio:

**Trascrizione:**
"Hello. We're excited to announce the new framework Data Piz AI."

**Riassunto del contenuto principale:**
L'audio annuncia con entusiasmo il lancio di un nuovo framework chiamato "Data Piz AI".)]


In [32]:
import os
import base64
from pathlib import Path
from dotenv import load_dotenv

from datapizzai.clients import ClientFactory
from datapizzai.memory import Memory
from datapizzai.type import ROLE, TextBlock, Media, MediaBlock

# Carica variabili d'ambiente
load_dotenv('../.env')

def create_mediablock_from_file(file_path: str) -> MediaBlock:
    """Crea un MediaBlock da un file immagine locale (base64)."""
    data = Path(file_path).read_bytes()
    image_b64 = base64.b64encode(data).decode('utf-8')
    ext = Path(file_path).suffix.lstrip('.').lower() or 'png'
    media = Media(
        extension=ext,
        media_type="image",
        source_type="base64",
        source=image_b64,
        detail="high",
    )
    return MediaBlock(media=media)

client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-4o",
)
memory = Memory()

# Primo turno: utente invia immagine con richiesta
image_block = create_mediablock_from_file("Client/Example.png")
memory.add_turn([TextBlock("Analizza questa foto, cosa vedi?"), image_block], ROLE.USER)
resp = client.invoke("", memory=memory)
memory.add_turn([TextBlock(resp.text)], ROLE.ASSISTANT) #Aggiungo risposta alla memoria

# Secondo turno: follow-up che si basa sull'immagine precedente
memory.add_turn([TextBlock("Quali miglioramenti consiglieresti?")], ROLE.USER)
resp = client.invoke("", memory=memory)

print(resp.text)

[TextBlock(content=Per migliorare l'immagine e l'ambiente generale, potresti considerare i seguenti suggerimenti:

1. **Organizzazione del Tavolo**: Sistemare i cavi e rimuovere eventuali oggetti non necessari per un aspetto più ordinato.

2. **Illuminazione**: Aggiungere più sorgenti di luce calda per rendere l'ambiente più accogliente e ridurre l'affaticamento visivo.

3. **Comfort**: Assicurarsi che le sedie siano ergonomiche per migliorare il comfort durante le lunghe ore di lavoro.

4. **Spazio**: Aumentare lo spazio tra i partecipanti per una maggiore comodità e mobilità.

5. **Piante**: Aggiungere qualche pianta per migliorare l'atmosfera e la qualità dell'aria.

6. **Pause Regolari**: Introdurre momenti di pausa per migliorare la produttività e il benessere del team.

Questi suggerimenti possono contribuire a creare un ambiente di lavoro più piacevole ed efficiente.)]


In [34]:
import os, json
from dotenv import load_dotenv
from datapizzai.clients import ClientFactory

load_dotenv()
client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5",
    temperature=1,
)

prompt = (
    "Fornisci un riepilogo progetto come JSON valido, senza testo extra.\n"
    "Schema: {\n"
    "  \"title\": string,\n"
    "  \"status\": one of [planned, in_progress, done],\n"
    "  \"tasks\": array of {\n"
    "    \"name\": string, \"owner\": string, \"eta_days\": integer\n"
    "  }\n"
    "}"
)

resp = client.invoke(prompt)
raw = resp.text

# Parsing lato client
data = json.loads(raw)
print("Titolo:", data["title"])  # es.: "Migrazione a microservizi"

Titolo: Sviluppo piattaforma e-commerce


In [35]:
data

{'title': 'Sviluppo piattaforma e-commerce',
 'status': 'in_progress',
 'tasks': [{'name': 'Analisi requisiti',
   'owner': 'Giulia Rossi',
   'eta_days': 5},
  {'name': 'Design architetturale', 'owner': 'Marco Bianchi', 'eta_days': 8},
  {'name': 'Implementazione backend', 'owner': 'Luca Verdi', 'eta_days': 12},
  {'name': 'Test end-to-end', 'owner': 'Sara Neri', 'eta_days': 6}]}

In [38]:
import os
from typing import List
from pydantic import BaseModel
from dotenv import load_dotenv
from datapizzai.clients import ClientFactory

# Definizione delle classi Pydantic per la struttura dati
class Task(BaseModel):
    name: str
    owner: str
    eta_days: int

class ProjectSummary(BaseModel):
    title: str
    status: str  # planned, in_progress, done
    tasks: List[Task]

load_dotenv()
client = ClientFactory.create(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-4o",
)

# Uso del metodo structured_response con classe Pydantic
response = client.structured_response(
    input="Riepiloga il piano progetto per il nuovo portale e-commerce",
    output_cls=ProjectSummary,
)

# Il risultato è disponibile come response.structured_data[0] 
structured = response.structured_data[0]
print("Titolo:", structured.title)
print("Status:", structured.status)
print("Tasks:")
for task in structured.tasks:
    print(f"  - {task.name} (owner: {task.owner}, ETA: {task.eta_days} giorni)")

Titolo: Piano Progetto per il Nuovo Portale E-commerce
Status: In Progresso
Tasks:
  - Analisi dei Requisiti (owner: Giovanni Rossi, ETA: 14 giorni)
  - Progettazione UI/UX (owner: Marta Bianchi, ETA: 21 giorni)
  - Sviluppo Backend (owner: Luca Verdi, ETA: 30 giorni)
  - Sviluppo Frontend (owner: Chiara Neri, ETA: 28 giorni)
  - Integrazione Sistemi di Pagamento (owner: Marco Ricci, ETA: 10 giorni)
  - Test e QA (owner: Sara Galli, ETA: 15 giorni)
  - Formazione del Personale (owner: Laura Conti, ETA: 7 giorni)
  - Lancio e Monitoraggio (owner: Paolo Moretti, ETA: 5 giorni)


## Tool

In [2]:
from datapizzai.tools import tool

@tool
def timer_tool(duration: str) -> str:
    """Imposta un timer (es. "5 minutes")."""
    # DO something (stub)
    return f"Timer impostato per {duration}"

In [7]:
from datapizzai.clients import ClientFactory
from dotenv import load_dotenv
import os

load_dotenv()
client = ClientFactory.create(
    provider="openai", 
    api_key=os.getenv("OPENAI_API_KEY"), 
    model="gpt-5",
    temperature=1
    )

response = client.invoke(
    "Set a timer for 5 minutes",
    tools=[timer_tool],
    tool_choice="auto"
)

print(response.text)  # Qualsiasi risposta testuale
for f_call in response.function_calls or []:
    # Esegui il tool locale con gli argomenti suggeriti
    result = timer_tool(**(f_call.arguments or {}))
    print("tool result:", result)


tool result: Timer impostato per 5 minutes


In [8]:
from datapizzai.tools import tool

@tool
def calcolatrice(expr: str) -> str:
    """Esegue calcoli semplici in modo sicuro (demo)."""
    try:
        allowed = set("0123456789+-*/(). ")
        if not set(expr) <= allowed:
            return "Errore: caratteri non validi"
        return str(eval(expr))
    except Exception as e:
        return f"Errore: {e}"

@tool
def cerca_informazioni(query: str) -> str:
    """Dummy search (esempio)."""
    return f"(risultati sintetici per: {query})"

In [9]:
# Client e Memory  
from datapizzai.clients import ClientFactory
from datapizzai.memory import Memory
from datapizzai.type import FunctionCallResultBlock, ROLE
from dotenv import load_dotenv
import os

load_dotenv()
client = ClientFactory.create(provider="openai", api_key=os.getenv("OPENAI_API_KEY"), model="gpt-4o")

tools = [calcolatrice, cerca_informazioni]
memory = Memory()

response = client.invoke(
    input="Calcola (25 * 4) + 10 e cerca informazioni su Python type hints",
    tools=tools,
    tool_choice="auto",
    memory=memory
)

# Esecuzione iterativa dei function call
while hasattr(response, "function_calls") and response.function_calls:
    # Aggiungi la risposta dell'assistant alla memoria
    memory.add_turn(response.content, ROLE.ASSISTANT)
    
    # Crea i risultati dei tool e aggiungili uno per volta alla memoria
    for f_call in response.function_calls:
        tool_name = f_call.name
        args = f_call.arguments or {}
        
        if tool_name == "calcolatrice":
            result = calcolatrice(**args)
        elif tool_name == "cerca_informazioni":
            result = cerca_informazioni(**args)
        else:
            result = f"Tool sconosciuto: {tool_name}"

        tool_result_block = FunctionCallResultBlock(
            id=f_call.id,
            tool=tool_name,
            result=result,
        )
        
        # Aggiungi ogni tool result come turn separato con ruolo TOOL
        memory.add_turn([tool_result_block], ROLE.TOOL)

    # Re-invoca con la memoria aggiornata
    response = client.invoke(
        input="",
        tools=tools,
        tool_choice="auto",
        memory=memory
    )

print(response.text)

Il risultato del calcolo \((25 \times 4) + 10\) è 110.

Per quanto riguarda "Python type hints", i risultati sintetici suggeriscono che i type hints in Python sono una funzionalità che consente ai programmatori di annotare il tipo di variabili, parametri delle funzioni e valori di ritorno, migliorando così la leggibilità e la manutenzione del codice. I type hints non sono obbligatori e Python continua ad essere un linguaggio dinamico, ma forniscono un modo per documentare le aspettative sui tipi ed aiutare con strumenti di analisi statica del codice.


In [20]:
from datapizzai.memory import Memory
from datapizzai.type import TextBlock, ROLE

def create_conversational_client():
    memory = Memory()
    client = ClientFactory.create(
        provider="openai",
        api_key=os.getenv("OPENAI_API_KEY"),
        model="gpt-4o",
    )
    return client, memory

# 3. Configura conversazione multi-turno
client, memory = create_conversational_client()
tools = [calcolatrice, cerca_informazioni]

def chat_turn(user_input, memory, client, tools):
    """Gestisce un singolo turno di conversazione con tools"""
    print(f"👤 Utente: {user_input}")
    
    # Aggiungi input utente alla memoria
    memory.add_turn([TextBlock(content=user_input)], ROLE.USER)
    
    # Prima chiamata al modello
    response = client.invoke(
        input="",  # Input vuoto perché usiamo la memory
        memory=memory,
        tools=tools,
        tool_choice="auto"
        # ❌ NON usare tool_results qui!
    )
    
    # Gestione iterativa dei function calls
    while hasattr(response, "function_calls") and response.function_calls:
        print("🔧 Esecuzione tool calls...")
        
        # Aggiungi la risposta dell'assistant alla memoria
        memory.add_turn(response.content, ROLE.ASSISTANT)
        
        # Esegui ogni function call
        for f_call in response.function_calls:
            print(f"   📞 {f_call.name}({f_call.arguments})")
            
            # Esegui il tool (il tuo codice esistente va bene)
            result = {
                "calcolatrice": calcolatrice,
                "cerca_informazioni": cerca_informazioni,
            }.get(f_call.name, lambda **_: f"Tool sconosciuto: {f_call.name}")(**(f_call.arguments or {}))
            
            print(f"   ✅ {result}")
            
            # Crea il blocco risultato
            tool_result_block = FunctionCallResultBlock(
                id=f_call.id, 
                tool=f_call.name, 
                result=result
            )
            
            # ✅ SOLUZIONE: Aggiungi alla memoria invece di usare tool_results
            memory.add_turn([tool_result_block], ROLE.TOOL)
        
        # ✅ Richiedi risposta finale senza tool_results
        response = client.invoke(
            input="",
            memory=memory,
            tools=tools,
            tool_choice="auto"
            # ❌ NON usare tool_results qui!
        )
    
    # Aggiungi la risposta finale alla memoria
    if response.text:
        memory.add_turn([TextBlock(content=response.text)], ROLE.ASSISTANT)
        print(f"🤖 Assistant: {response.text}")

# 4. Esempio di conversazione multi-turno
conversation = [
    "Ciao! Sono Mirko, sto lavorando su un progetto AI",
    "Cerca informazioni sui framework Python per AI",
    "Calcola il costo se spendo 500€ al mese per 2 anni",
    "Ricordi il mio nome e cosa sto facendo?"
]

for user_input in conversation:
    chat_turn(user_input, memory, client, tools)
    print()  # Spazio tra turni

# 5. Statistiche conversazione
print(f"📊 Turni totali: {len(memory.memory)}")
print(f"💬 Blocchi totali: {len(list(memory.iter_blocks()))}")

👤 Utente: Ciao! Sono Mirko, sto lavorando su un progetto AI
🤖 Assistant: Ciao Mirko! È fantastico sapere che stai lavorando su un progetto AI. Come posso aiutarti oggi? Hai domande o qualcosa di specifico su cui stai lavorando e su cui ti serve supporto?

👤 Utente: Cerca informazioni sui framework Python per AI
🔧 Esecuzione tool calls...
   📞 cerca_informazioni({'query': 'framework Python per AI'})
   ✅ (risultati sintetici per: framework Python per AI)
🤖 Assistant: Ecco alcune informazioni sui framework Python più popolari per l'intelligenza artificiale:

1. **TensorFlow**: Sviluppato da Google, TensorFlow è uno dei framework più usati per il machine learning e l'intelligenza artificiale. È flessibile e si adatta bene a diversi tipi di modelli di apprendimento automatico.

2. **PyTorch**: Creato da Facebook, PyTorch è apprezzato per la sua facilità d'uso e per la sua capacità di eseguire calcoli dinamici. È particolarmente popolare nella comunità di ricerca.

3. **Keras**: Un'interfac

In [18]:
import os
from dotenv import load_dotenv
from datapizzai.clients import ClientFactory
from datapizzai.tools.google import google_search_tool

load_dotenv()

# Assicurati di avere GOOGLE_API_KEY nel file .env
client = ClientFactory.create(
    provider="google",
    api_key=os.getenv("GOOGLE_API_KEY"),
    model="gemini-2.0-flash",
)

response = client.invoke("Quando iniziano le olimpiadi invernali?", tools=[google_search_tool])

response.text

'Le Olimpiadi invernali di Milano Cortina 2026 inizieranno il 6 febbraio 2026 e dureranno fino al 22 febbraio 2026. Tuttavia, le gare inizieranno il 4 febbraio 2026.\n'